# Data Exploration

### Importing necessary libraries

In [ ]:
!pip install datasist
!pip install category_encoders
!pip install lightgbm
!pip install xgboost

In [4]:
import pandas as pd
import numpy as np
import plotly.express as px
from datasist.structdata import detect_outliers

In [422]:
df = pd.read_csv("adult.csv")

In [423]:
df.head()

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K


In [424]:
df['income'].value_counts(normalize=True)

income
<=50K    0.760718
>50K     0.239282
Name: proportion, dtype: float64

***Columns Interpretation***
| Column           | Description |
|------------------|-------------|
| **age** | Age |
| **workclass** | A general term indicating the employment status of an individual. |
| **fnlwgt** | Final weight, representing the number of individuals that this row represents (a representative sample). |
| **education** | Highest level of education achieved by an individual. |
| **education.num** | Highest level of education achieved by an individual in numerical form. |
| **marital.status** | Marital status of an individual. Note that `Married-civ-spouse` refers to a civilian spouse, and `Married-AF-spouse` refers to a spouse in the Armed Forces. |
| **occupation** | General type of occupation of an individual. |
| **relationship** | Relationship of this individual with others, for example, spouse (`Husband`). Each data point has only one relationship. |
| **race** | Race |
| **sex** | Biological sex of an individual. |
| **capital.gain** | Capital gains of an individual. |
| **capital.loss** | Capital losses of an individual. |
| **hours.per.week** | Number of hours the individual reported working per week. |
| **native.country** | Country of origin. |
| **income** | Income, less than or equal to $50,000 (`<=50K`) or more than that (`>50K`). |

In [425]:
df.columns

Index(['age', 'workclass', 'fnlwgt', 'education', 'educational-num',
       'marital-status', 'occupation', 'relationship', 'race', 'gender',
       'capital-gain', 'capital-loss', 'hours-per-week', 'native-country',
       'income'],
      dtype='object')

In [426]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   age              48842 non-null  int64 
 1   workclass        48842 non-null  object
 2   fnlwgt           48842 non-null  int64 
 3   education        48842 non-null  object
 4   educational-num  48842 non-null  int64 
 5   marital-status   48842 non-null  object
 6   occupation       48842 non-null  object
 7   relationship     48842 non-null  object
 8   race             48842 non-null  object
 9   gender           48842 non-null  object
 10  capital-gain     48842 non-null  int64 
 11  capital-loss     48842 non-null  int64 
 12  hours-per-week   48842 non-null  int64 
 13  native-country   48842 non-null  object
 14  income           48842 non-null  object
dtypes: int64(6), object(9)
memory usage: 5.6+ MB


In [427]:
# Check duplicates
dup = df[df.duplicated() == True]
dup.shape

(52, 15)

In [428]:
df.drop_duplicates(inplace= True, ignore_index= True)

In [429]:
df.duplicated().sum()

np.int64(0)

In [430]:
# Check Missing Values by count
df.isna().sum()

age                0
workclass          0
fnlwgt             0
education          0
educational-num    0
marital-status     0
occupation         0
relationship       0
race               0
gender             0
capital-gain       0
capital-loss       0
hours-per-week     0
native-country     0
income             0
dtype: int64

In [431]:
# Check Summary statistics for Numerical Columns
df.describe().round(2)

,age,fnlwgt,educational-num,capital-gain,capital-loss,hours-per-week
count,48790.00,48790.00,48790.00,48790.00,48790.00,48790.00
mean,38.65,189669.00,10.08,1080.22,87.60,40.43
std,13.71,105617.23,2.57,7455.91,403.21,12.39
min,17.00,12285.00,1.00,0.00,0.00,1.00
25%,28.00,117555.00,9.00,0.00,0.00,40.00
50%,37.00,178138.50,10.00,0.00,0.00,40.00
75%,48.00,237606.25,12.00,0.00,0.00,45.00
max,90.00,1490400.00,16.00,99999.00,4356.00,99.00


In [432]:
#Check outliers in all columns
num_cols = df.select_dtypes(exclude = 'object')
for col in num_cols:
  outlier_indices = detect_outliers(data= df, n= 0, features= [col])
  print(f'{col}: Number of Outliers: {len(outlier_indices)}')
  print(f'Percentage of outliers: {round(len(outlier_indices)/df.shape[0] * 100, 2)}')
  print('-' * 100)

age: Number of Outliers: 215
Percentage of outliers: 0.44
----------------------------------------------------------------------------------------------------
fnlwgt: Number of Outliers: 1453
Percentage of outliers: 2.98
----------------------------------------------------------------------------------------------------
educational-num: Number of Outliers: 1787
Percentage of outliers: 3.66
----------------------------------------------------------------------------------------------------
capital-gain: Number of Outliers: 4035
Percentage of outliers: 8.27
----------------------------------------------------------------------------------------------------
capital-loss: Number of Outliers: 2282
Percentage of outliers: 4.68
----------------------------------------------------------------------------------------------------
hours-per-week: Number of Outliers: 13486
Percentage of outliers: 27.64
------------------------------------------------------------------------------------------------

Will not drop the outliers in hours-per-week column because it represents a notable percentage and may influence model prectivtivity later

In [433]:
# Check Summary statistics for Categorical Columns
df.describe(include= ['object'])

,workclass,education,marital-status,occupation,relationship,race,gender,native-country,income
count,48790,48790,48790,48790,48790,48790,48790,48790,48790
unique,9,16,7,15,6,5,2,42,2
top,Private,HS-grad,Married-civ-spouse,Prof-specialty,Husband,White,Male,United-States,<=50K
freq,33860,15770,22366,6165,19703,41714,32614,43792,37109


In [434]:
df.columns

Index(['age', 'workclass', 'fnlwgt', 'education', 'educational-num',
       'marital-status', 'occupation', 'relationship', 'race', 'gender',
       'capital-gain', 'capital-loss', 'hours-per-week', 'native-country',
       'income'],
      dtype='object')

In [435]:
# Drop unnecessary columns
df.drop(['educational-num'], axis= 1, inplace= True)

In [436]:
df.duplicated().sum()

np.int64(0)

In [437]:
cat_cols = df.select_dtypes(include= 'object').columns
cat_cols

Index(['workclass', 'education', 'marital-status', 'occupation',
       'relationship', 'race', 'gender', 'native-country', 'income'],
      dtype='object')

In [438]:
for col in cat_cols:

    print(col)
    print(df[col].nunique())
    print(df[col].unique())
    print('-' * 100)

workclass
9
['Private' 'Local-gov' '?' 'Self-emp-not-inc' 'Federal-gov' 'State-gov'
 'Self-emp-inc' 'Without-pay' 'Never-worked']
----------------------------------------------------------------------------------------------------
education
16
['11th' 'HS-grad' 'Assoc-acdm' 'Some-college' '10th' 'Prof-school'
 '7th-8th' 'Bachelors' 'Masters' 'Doctorate' '5th-6th' 'Assoc-voc' '9th'
 '12th' '1st-4th' 'Preschool']
----------------------------------------------------------------------------------------------------
marital-status
7
['Never-married' 'Married-civ-spouse' 'Widowed' 'Divorced' 'Separated'
 'Married-spouse-absent' 'Married-AF-spouse']
----------------------------------------------------------------------------------------------------
occupation
15
['Machine-op-inspct' 'Farming-fishing' 'Protective-serv' '?'
 'Other-service' 'Prof-specialty' 'Craft-repair' 'Adm-clerical'
 'Exec-managerial' 'Tech-support' 'Sales' 'Priv-house-serv'
 'Transport-moving' 'Handlers-cleaners' 'Armed-For

In [439]:
for col in cat_cols:

    print(df[col].value_counts(normalize=True) * 100)
    print('-' * 100)

workclass
Private             69.399467
Self-emp-not-inc     7.913507
Local-gov            6.427547
?                    5.728633
State-gov            4.060258
Self-emp-inc         3.472023
Federal-gov          2.935028
Without-pay          0.043042
Never-worked         0.020496
Name: proportion, dtype: float64
----------------------------------------------------------------------------------------------------
education
HS-grad         32.322197
Some-college    22.264808
Bachelors       16.423447
Masters          5.443738
Assoc-voc        4.222177
11th             3.713876
Assoc-acdm       3.281410
10th             2.846895
7th-8th          1.955319
Prof-school      1.709367
9th              1.549498
12th             1.342488
Doctorate        1.217463
5th-6th          1.039147
1st-4th          0.502152
Preschool        0.166018
Name: proportion, dtype: float64
----------------------------------------------------------------------------------------------------
marital-status
Married-civ

In [440]:
pd.crosstab(df['relationship'], df['gender'], normalize='index', dropna = False).round(5) * 100


gender,Female,Male
relationship,,
Husband,0.005,99.995
Not-in-family,46.675,53.325
Other-relative,45.750,54.250
Own-child,44.524,55.476
Unmarried,76.639,23.361
Wife,99.871,0.129


In [441]:
df[(df['relationship'] == 'Husband') & (df['gender'] != 'Male')]

,age,workclass,fnlwgt,education,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
23379,34,Private,175878,HS-grad,Married-civ-spouse,Sales,Husband,White,Female,0,0,40,United-States,<=50K


In [442]:
df[(df['relationship'] == 'Wife') & (df['gender'] != 'Female')]

,age,workclass,fnlwgt,education,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
5660,64,Local-gov,152172,10th,Married-civ-spouse,Machine-op-inspct,Wife,White,Male,0,0,40,?,<=50K
16851,29,Private,350162,Bachelors,Married-civ-spouse,Exec-managerial,Wife,White,Male,0,0,40,United-States,>50K
43381,36,Private,74791,Bachelors,Married-civ-spouse,Sales,Wife,White,Male,0,0,60,?,<=50K


In [443]:
pd.crosstab(df['marital-status'], df['relationship'], normalize='index', dropna = False).round(2) * 100

relationship,Husband,Not-in-family,Other-relative,Own-child,Unmarried,Wife
marital-status,,,,,,
Divorced,0.0,55.0,3.0,7.0,36.0,0.0
Married-AF-spouse,32.0,0.0,3.0,3.0,0.0,62.0
Married-civ-spouse,88.0,0.0,1.0,1.0,0.0,10.0
Married-spouse-absent,0.0,52.0,9.0,10.0,29.0,0.0
Never-married,0.0,44.0,6.0,42.0,8.0,0.0
Separated,0.0,42.0,5.0,10.0,44.0,0.0
Widowed,0.0,56.0,5.0,2.0,38.0,0.0


In [444]:
pd.crosstab(df['workclass'], df['occupation'], normalize='index', dropna = False).round(2) * 100


occupation,?,Adm-clerical,Armed-Forces,Craft-repair,Exec-managerial,Farming-fishing,Handlers-cleaners,Machine-op-inspct,Other-service,Priv-house-serv,Prof-specialty,Protective-serv,Sales,Tech-support,Transport-moving
workclass,,,,,,,,,,,,,,,
?,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Federal-gov,0.0,34.0,1.0,6.0,19.0,1.0,3.0,1.0,4.0,0.0,18.0,3.0,1.0,7.0,3.0
Local-gov,0.0,13.0,0.0,7.0,11.0,1.0,2.0,1.0,10.0,0.0,34.0,14.0,1.0,2.0,5.0
Never-worked,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Private,0.0,12.0,0.0,14.0,12.0,2.0,6.0,8.0,12.0,1.0,10.0,1.0,13.0,3.0,6.0
Self-emp-inc,0.0,3.0,0.0,10.0,36.0,5.0,0.0,1.0,2.0,0.0,14.0,0.0,25.0,1.0,2.0
Self-emp-not-inc,0.0,2.0,0.0,21.0,15.0,17.0,1.0,2.0,7.0,0.0,15.0,0.0,15.0,1.0,5.0
State-gov,0.0,19.0,0.0,5.0,14.0,1.0,1.0,1.0,10.0,0.0,32.0,9.0,1.0,4.0,3.0
Without-pay,0.0,14.0,0.0,5.0,5.0,38.0,10.0,10.0,10.0,0.0,0.0,0.0,5.0,0.0,5.0


# Data Cleaning

Data Cleaning:
- fixing column names
- renaming the categories into interpretable words
- drop missings in native-country & group it by regions
- drop/impute missings in workclass and occupation: 'Never-worked' workclass --> their occupation is no job
- group Husband & Wife in 'relationship' into married and drop Male Wives & Female Husbands

In [445]:
#fixing column names
df.columns = df.columns.str.replace('-', '_')
df.columns

Index(['age', 'workclass', 'fnlwgt', 'education', 'marital_status',
       'occupation', 'relationship', 'race', 'gender', 'capital_gain',
       'capital_loss', 'hours_per_week', 'native_country', 'income'],
      dtype='object')

In [446]:
df = df.replace('?', np.nan)

In [447]:
df.isna().mean() *100

age               0.000000
workclass         5.728633
fnlwgt            0.000000
education         0.000000
marital_status    0.000000
occupation        5.749129
relationship      0.000000
race              0.000000
gender            0.000000
capital_gain      0.000000
capital_loss      0.000000
hours_per_week    0.000000
native_country    1.754458
income            0.000000
dtype: float64

In [448]:
workclass_map = {'Private': 'Private' ,
                  'Local-gov':"Local Government",
                  'Self-emp-not-inc': 'Self-Employed (Not Incorporated)' ,
                  'Federal-gov': 'Federal Government' ,
                  'State-gov': 'State Government' ,
                  'Self-emp-inc': 'Self-Employed (Incorporated)' ,
                  'Without-pay': 'Without Pay' ,
                  'Never-worked': 'Never Worked'}

df['workclass'] = df['workclass'].map(workclass_map)
df['workclass'].value_counts()


workclass
Private                             33860
Self-Employed (Not Incorporated)     3861
Local Government                     3136
State Government                     1981
Self-Employed (Incorporated)         1694
Federal Government                   1432
Without Pay                            21
Never Worked                           10
Name: count, dtype: int64

In [449]:
df.isna().sum()

age                  0
workclass         2795
fnlwgt               0
education            0
marital_status       0
occupation        2805
relationship         0
race                 0
gender               0
capital_gain         0
capital_loss         0
hours_per_week       0
native_country     856
income               0
dtype: int64

In [450]:
education_map = {
    '11th': '11th Grade',
    'HS-grad': 'High School Graduate',
    'Assoc-acdm': "Associate's Degree (Academic)",
    'Some-college': 'College',
    '10th': '10th Grade',
    'Prof-school': 'Professional School',
    '7th-8th': '7th to 8th Grade',
    'Bachelors': "Bachelor's Degree",
    'Masters': "Master's Degree",
    'Doctorate': "Doctorate/PhD",
    '5th-6th': '5th to 6th Grade',
    'Assoc-voc': "Associate's Degree (Vocational)",
    '9th': '9th Grade',
    '12th': '12th Grade',
    '1st-4th': '1st to 4th Grade',
    'Preschool': 'Preschool'
}

df['education'] = df['education'].map(education_map)
df['education'].value_counts()


education
High School Graduate               15770
College                            10863
Bachelor's Degree                   8013
Master's Degree                     2656
Associate's Degree (Vocational)     2060
11th Grade                          1812
Associate's Degree (Academic)       1601
10th Grade                          1389
7th to 8th Grade                     954
Professional School                  834
9th Grade                            756
12th Grade                           655
Doctorate/PhD                        594
5th to 6th Grade                     507
1st to 4th Grade                     245
Preschool                             81
Name: count, dtype: int64

In [451]:
df.isna().sum()

age                  0
workclass         2795
fnlwgt               0
education            0
marital_status       0
occupation        2805
relationship         0
race                 0
gender               0
capital_gain         0
capital_loss         0
hours_per_week       0
native_country     856
income               0
dtype: int64

In [452]:
marital_status_map = {
    'Never-married': 'Never Married',
    'Married-civ-spouse': 'Married (Civilian Spouse)',
    'Widowed': 'Widowed',
    'Divorced': 'Divorced',
    'Separated': 'Separated',
    'Married-spouse-absent': 'Married, Spouse Absent',
    'Married-AF-spouse': 'Married (Armed Forces Spouse)'
}

# Apply mapping to DataFrame
df['marital_status'] = df['marital_status'].map(marital_status_map)
df['marital_status'].value_counts()


marital_status
Married (Civilian Spouse)        22366
Never Married                    16082
Divorced                          6630
Separated                         1530
Widowed                           1518
Married, Spouse Absent             627
Married (Armed Forces Spouse)       37
Name: count, dtype: int64

In [453]:

occupation_map = {
    'Machine-op-inspct': 'Machine Operator / Inspector',
    'Farming-fishing': 'Farming / Fishing',
    'Protective-serv': 'Protective Service',
    'Other-service': 'Other Service',
    'Prof-specialty': 'Professional Specialty',
    'Craft-repair': 'Craft / Repair',
    'Adm-clerical': 'Administrative / Clerical',
    'Exec-managerial': 'Executive / Managerial',
    'Tech-support': 'Technical Support',
    'Sales': 'Sales',
    'Priv-house-serv': 'Private House Service',
    'Transport-moving': 'Transport / Moving',
    'Handlers-cleaners': 'Handlers / Cleaners',
    'Armed-Forces': 'Armed Forces',
}

# Apply mapping to DataFrame
df['occupation'] = df['occupation'].map(occupation_map)
df['occupation'].value_counts()

occupation
Professional Specialty          6165
Craft / Repair                  6102
Executive / Managerial          6082
Administrative / Clerical       5606
Sales                           5501
Other Service                   4919
Machine Operator / Inspector    3017
Transport / Moving              2355
Handlers / Cleaners             2071
Farming / Fishing               1485
Technical Support               1445
Protective Service               982
Private House Service            240
Armed Forces                      15
Name: count, dtype: int64

In [454]:
df.isna().mean() *100

age               0.000000
workclass         5.728633
fnlwgt            0.000000
education         0.000000
marital_status    0.000000
occupation        5.749129
relationship      0.000000
race              0.000000
gender            0.000000
capital_gain      0.000000
capital_loss      0.000000
hours_per_week    0.000000
native_country    1.754458
income            0.000000
dtype: float64

In [455]:
#grouping countries into regions
country_to_region = {
    # US & Canada
    'United-States': 'US & Canada',
    'Canada': 'US & Canada',
    'Outlying-US(Guam-USVI-etc)': 'US & Canada',

    # East Asia & Pacific
    'Philippines': 'East Asia & Pacific',
    'Thailand': 'East Asia & Pacific',
    'Vietnam': 'East Asia & Pacific',
    'Cambodia': 'East Asia & Pacific',
    'Laos': 'East Asia & Pacific',
    'Japan': 'East Asia & Pacific',
    'Taiwan': 'East Asia & Pacific',
    'China': 'East Asia & Pacific',
    'Hong': 'East Asia & Pacific',

    # Europe & Central Asia
    'Ireland': 'Europe & Central Asia',
    'Germany': 'Europe & Central Asia',
    'Poland': 'Europe & Central Asia',
    'England': 'Europe & Central Asia',
    'Italy': 'Europe & Central Asia',
    'Portugal': 'Europe & Central Asia',
    'Scotland': 'Europe & Central Asia',
    'Yugoslavia': 'Europe & Central Asia',
    'Hungary': 'Europe & Central Asia',
    'Greece': 'Europe & Central Asia',
    'France': 'Europe & Central Asia',
    'Holand-Netherlands': 'Europe & Central Asia',

    # Latin America & Caribbean
    'Peru': 'Latin America & Caribbean',
    'Guatemala': 'Latin America & Caribbean',
    'Mexico': 'Latin America & Caribbean',
    'Dominican-Republic': 'Latin America & Caribbean',
    'Haiti': 'Latin America & Caribbean',
    'El-Salvador': 'Latin America & Caribbean',
    'Puerto-Rico': 'Latin America & Caribbean',
    'Columbia': 'Latin America & Caribbean',
    'Cuba': 'Latin America & Caribbean',
    'Nicaragua': 'Latin America & Caribbean',
    'Honduras': 'Latin America & Caribbean',
    'Jamaica': 'Latin America & Caribbean',
    'Ecuador': 'Latin America & Caribbean',
    'Trinadad&Tobago': 'Latin America & Caribbean',

    # Middle East
    'Iran': 'Middle East',

    # South Asia
    'India': 'South Asia',

    # Handle unknowns
    'South': np.nan,
    '?': np.nan
}

df['region'] = df['native_country'].replace(country_to_region)


In [456]:
df['region'].value_counts(normalize= True, dropna=False) * 100

region
US & Canada                  90.176266
Latin America & Caribbean     4.226276
NaN                           1.990162
Europe & Central Asia         1.598688
East Asia & Pacific           1.578192
South Asia                    0.309490
Middle East                   0.120926
Name: proportion, dtype: float64

In [457]:
#drop missings in the new region column

df = df.dropna(subset= ['region']).reset_index(drop= True)
df['region'].value_counts(normalize= True, dropna=False) * 100

region
US & Canada                  92.007361
Latin America & Caribbean     4.312094
Europe & Central Asia         1.631151
East Asia & Pacific           1.610239
South Asia                    0.315774
Middle East                   0.123382
Name: proportion, dtype: float64

In [458]:
#dropping original native-country column
df.drop('native_country', axis= 1, inplace= True)
df.head()

,age,workclass,fnlwgt,education,marital_status,occupation,relationship,race,gender,capital_gain,capital_loss,hours_per_week,income,region
0,25,Private,226802,11th Grade,Never Married,Machine Operator / Inspector,Own-child,Black,Male,0,0,40,<=50K,US & Canada
1,38,Private,89814,High School Graduate,Married (Civilian Spouse),Farming / Fishing,Husband,White,Male,0,0,50,<=50K,US & Canada
2,28,Local Government,336951,Associate's Degree (Academic),Married (Civilian Spouse),Protective Service,Husband,White,Male,0,0,40,>50K,US & Canada
3,44,Private,160323,College,Married (Civilian Spouse),Machine Operator / Inspector,Husband,Black,Male,7688,0,40,>50K,US & Canada
4,18,NaN,103497,College,Never Married,NaN,Own-child,White,Female,0,0,30,<=50K,US & Canada


In [459]:
#drop/impute missings in workclass and occupation
df[df['workclass'] == 'Never Worked'][['occupation', 'workclass']]

,occupation,workclass
8625,NaN,Never Worked
11391,NaN,Never Worked
13631,NaN,Never Worked
21208,NaN,Never Worked
26571,NaN,Never Worked
30413,NaN,Never Worked
35865,NaN,Never Worked
38691,NaN,Never Worked
47570,NaN,Never Worked
47579,NaN,Never Worked


In [460]:
#'Never-worked' workclass --> their occupation is no job
df.loc[df['workclass'] == 'Never Worked', 'occupation'] = 'No Occupation'
df[df['workclass'] == 'Never Worked'][['occupation', 'workclass']]

,occupation,workclass
8625,No Occupation,Never Worked
11391,No Occupation,Never Worked
13631,No Occupation,Never Worked
21208,No Occupation,Never Worked
26571,No Occupation,Never Worked
30413,No Occupation,Never Worked
35865,No Occupation,Never Worked
38691,No Occupation,Never Worked
47570,No Occupation,Never Worked
47579,No Occupation,Never Worked


In [461]:
df.isna().mean() * 100

age               0.000000
workclass         5.719484
fnlwgt            0.000000
education         0.000000
marital_status    0.000000
occupation        5.719484
relationship      0.000000
race              0.000000
gender            0.000000
capital_gain      0.000000
capital_loss      0.000000
hours_per_week    0.000000
income            0.000000
region            0.000000
dtype: float64

In [462]:
#drop Male Wives & Female Husbands

df = df[~((df['gender'] == 'Male') & (df['relationship'] == 'Wife'))]
df = df[~((df['gender'] == 'Female') & (df['relationship'] == 'Husband'))]


In [463]:
#group Husband & Wife in 'relationship' into 'Married'

df['relationship'] = df['relationship'].replace(['Husband', 'Wife'], 'Married')
df['relationship'].unique()

array(['Own-child', 'Married', 'Not-in-family', 'Unmarried',
       'Other-relative'], dtype=object)

In [464]:
df = df.reset_index(drop=True)

In [465]:
df['income'].value_counts(normalize= True) * 100

income
<=50K    76.075454
>50K     23.924546
Name: proportion, dtype: float64

Feature Engineering Overview

We'll create:

1. `net_capital` — combines gains and losses
2. `has_captial_activity` — categorizes Yes or No based on `capital_gain` and `capital_loss`
3. Categorize `hours.per.week` into: part-time, full-time, overtime
---


In [466]:
#Computing a column for net capital

df['net_capital'] = df['capital_gain'] - df['capital_loss']
df.sample(10)

,age,workclass,fnlwgt,education,marital_status,occupation,relationship,race,gender,capital_gain,capital_loss,hours_per_week,income,region,net_capital
3814,34,Private,170017,High School Graduate,Divorced,Craft / Repair,Own-child,White,Male,0,0,40,<=50K,US & Canada,0
42124,37,Private,209214,5th to 6th Grade,Never Married,Machine Operator / Inspector,Unmarried,White,Female,0,0,40,<=50K,Latin America & Caribbean,0
3141,26,Private,192652,Bachelor's Degree,Never Married,Executive / Managerial,Not-in-family,White,Male,0,0,20,<=50K,US & Canada,0
21234,42,Self-Employed (Not Incorporated),207392,High School Graduate,Married (Civilian Spouse),Sales,Married,White,Male,0,0,12,<=50K,US & Canada,0
29719,25,Private,212311,College,Never Married,Administrative / Clerical,Unmarried,Black,Female,0,0,40,<=50K,US & Canada,0
26839,47,Private,280483,High School Graduate,Separated,Craft / Repair,Unmarried,Black,Female,0,0,40,<=50K,US & Canada,0
26852,47,Local Government,29819,Master's Degree,Married (Civilian Spouse),Executive / Managerial,Married,Black,Male,0,1977,50,>50K,US & Canada,-1977
40972,34,State Government,20057,College,"Married, Spouse Absent",Administrative / Clerical,Unmarried,Asian-Pac-Islander,Female,0,0,38,<=50K,East Asia & Pacific,0
40124,46,Private,117059,Associate's Degree (Vocational),Married (Civilian Spouse),Transport / Moving,Married,Amer-Indian-Eskimo,Male,0,0,60,<=50K,US & Canada,0
39379,60,Self-Employed (Incorporated),210827,Bachelor's Degree,Married (Civilian Spouse),Professional Specialty,Married,White,Male,7688,0,40,>50K,US & Canada,7688


In [467]:
#Checking for Capital activity

df['has_capital_activity'] = ((df['capital_gain'] > 0) | (df['capital_loss'] > 0)).astype(int)
df.sample(10)

,age,workclass,fnlwgt,education,marital_status,occupation,relationship,race,gender,capital_gain,capital_loss,hours_per_week,income,region,net_capital,has_capital_activity
7270,28,Private,132750,High School Graduate,Divorced,Other Service,Unmarried,Black,Female,0,0,20,<=50K,US & Canada,0,0
21219,29,Private,148550,High School Graduate,Married (Civilian Spouse),Craft / Repair,Married,White,Male,0,0,40,<=50K,US & Canada,0,0
31536,35,Private,61343,Bachelor's Degree,Married (Civilian Spouse),Sales,Married,White,Male,0,0,40,>50K,US & Canada,0,0
38017,28,Local Government,168524,Master's Degree,Married (Civilian Spouse),Professional Specialty,Married,White,Female,7688,0,35,>50K,US & Canada,7688,1
47706,61,Private,190682,High School Graduate,Widowed,Craft / Repair,Not-in-family,Black,Female,0,1669,50,<=50K,US & Canada,-1669,1
23070,33,Private,101352,Professional School,Never Married,Professional Specialty,Not-in-family,White,Female,0,0,50,>50K,US & Canada,0,0
30941,32,Private,174704,11th Grade,Never Married,Other Service,Not-in-family,Black,Male,0,0,40,<=50K,US & Canada,0,0
26932,36,Self-Employed (Not Incorporated),188972,Doctorate/PhD,Separated,Professional Specialty,Unmarried,White,Female,0,0,10,<=50K,US & Canada,0,0
99,22,Private,212261,College,Never Married,Transport / Moving,Own-child,Black,Male,0,0,39,<=50K,US & Canada,0,0
32679,21,Private,155818,High School Graduate,Never Married,Other Service,Own-child,White,Female,0,0,20,<=50K,US & Canada,0,0


In [468]:
#Full_time vs Part_time vs Overtime

conditions = [
    df['hours_per_week'] < 35,
    df['hours_per_week'].between(35, 45, inclusive='both'),
    df['hours_per_week'] > 45
]

choices = ['Part_time', 'Full_time', 'Overtime']

df['employment_type'] = np.select(conditions, choices, default='Unknown')

df.sample(10)

,age,workclass,fnlwgt,education,marital_status,occupation,relationship,race,gender,capital_gain,capital_loss,hours_per_week,income,region,net_capital,has_capital_activity,employment_type
37938,21,Private,265434,College,Never Married,Professional Specialty,Own-child,White,Female,0,0,30,<=50K,US & Canada,0,0,Part_time
6304,37,Local Government,188612,Master's Degree,Married (Civilian Spouse),Executive / Managerial,Married,White,Male,0,0,50,>50K,US & Canada,0,0,Overtime
18701,26,Private,34402,Bachelor's Degree,Never Married,Administrative / Clerical,Not-in-family,White,Male,0,0,45,<=50K,US & Canada,0,0,Full_time
39818,44,Self-Employed (Not Incorporated),361280,College,Married (Civilian Spouse),Sales,Married,Asian-Pac-Islander,Male,0,0,80,>50K,East Asia & Pacific,0,0,Overtime
15717,44,Private,212894,10th Grade,Married (Civilian Spouse),Machine Operator / Inspector,Married,White,Male,0,0,40,<=50K,Europe & Central Asia,0,0,Full_time
23925,38,Local Government,82880,Bachelor's Degree,Married (Civilian Spouse),Professional Specialty,Married,White,Female,0,0,15,<=50K,US & Canada,0,0,Part_time
38748,33,Private,203463,High School Graduate,Divorced,Other Service,Own-child,White,Female,0,0,40,<=50K,US & Canada,0,0,Full_time
35016,36,Private,114059,Bachelor's Degree,Married (Civilian Spouse),Executive / Managerial,Married,White,Male,0,0,45,<=50K,US & Canada,0,0,Full_time
29076,27,Private,198346,Bachelor's Degree,Divorced,Administrative / Clerical,Not-in-family,White,Female,0,0,35,<=50K,US & Canada,0,0,Full_time
2816,31,Private,114691,Bachelor's Degree,Never Married,Sales,Not-in-family,White,Male,0,0,48,<=50K,US & Canada,0,0,Overtime


In [469]:
df.to_csv('data.csv', index=False)

In [470]:
df.isna().mean() *100

age                     0.000000
workclass               5.719723
fnlwgt                  0.000000
education               0.000000
marital_status          0.000000
occupation              5.719723
relationship            0.000000
race                    0.000000
gender                  0.000000
capital_gain            0.000000
capital_loss            0.000000
hours_per_week          0.000000
income                  0.000000
region                  0.000000
net_capital             0.000000
has_capital_activity    0.000000
employment_type         0.000000
dtype: float64

# Visualization & Analysis

In [471]:
!pip install -q streamlit

In [398]:
%%writefile Income_Analysis.py

import streamlit as st
import pandas as pd
import plotly.express as px


st.set_page_config(
    page_title="Adult Income Data Explorer", layout="wide")

st.title("💼 Adult Income Data Explorer")
st.markdown("""Welcome!""")

# -------------------------------
# Load and clean data
# -------------------------------
df = pd.read_csv("data.csv")
# Strip any hidden spaces in column names
df.columns = df.columns.str.strip()

st.header("Dataset Overview")
st.write(df.head())
st.write(f"Dataset shape: {df.shape}")

#Univariate Analysis

st.header("Univariate Analysis")
st.subheader("Age Distribution")
fig1 = px.histogram(df, x='age', nbins=30, title="Age Distribution")
st.plotly_chart(fig1, use_container_width=True)

st.subheader("Educational Levels")
fig2 = px.bar(df['education'].value_counts().reset_index(),
              x='education', y='count', title="Education Count")
st.plotly_chart(fig2, use_container_width=True)

st.subheader("Region of Origin Distribution")
fig3 = px.bar(df['region'].value_counts().reset_index(),
              x='region', y='count', title="Region Count")
st.plotly_chart(fig3, use_container_width=True)

st.subheader("Weekly Hours Worked")
fig4 = px.histogram(df, x='hours_per_week', nbins=30, title="Weekly Hours Distribution")
st.plotly_chart(fig4, use_container_width=True)

st.subheader("Gender Distribution")
fig11 = px.pie(df, names='gender', title="Gender Distribution", color='gender',
             color_discrete_map={'Male':"#313DEC", '    Female  ':"#3BDAEF"})
st.plotly_chart(fig11, use_container_width=True)


#Bivariate Analysis — Income Disparities

st.header("Bivariate Analysis — Income Disparities")
st.subheader("Income by Education")
education_order = ['Preschool', '1st to 4th Grade', '5th to 6th Grade', '7th to 8th Grade', '9th Grade', 
                   '10th Grade', '11th Grade', '12th Grade',
                   'High School Graduate', 'College',  "Associate's Degree (Academic)",
                   "Associate's Degree (Vocational)", "Bachelor's Degree",  "Master's Degree",
                   'Professional School', 'Doctorate/PhD']

fig7 = px.bar(df, x='education', color='income', barmode='group', title="Income by Education", 
              category_orders={'education': education_order})
st.plotly_chart(fig7, use_container_width=True)

st.subheader("Capital Activity by Income")
cap_income = df.groupby(['income', 'has_capital_activity']).size().reset_index(name='count')
fig5 = px.pie(cap_income, names='has_capital_activity', values='count', 
                     title="Capital Activity by Income",
                     color='has_capital_activity',
                     color_discrete_map={0:'#636EFA', 1:'#EF553B'},
                     facet_col='income')
st.plotly_chart(fig5, use_container_width=True)

st.subheader("Income by Gender")
fig8 = px.bar(df, x='gender', color='income', barmode='group', title="Income by Gender")
st.plotly_chart(fig8, use_container_width=True)


#Multivariate Analysis — Gender × Education × Income

st.header("Intersectional Trends — Gender × Education × Income")
st.subheader("Income by Education and Gender")
fig10 = px.bar(df, x='education', color='income', facet_col='gender',
               title="Income by Education and Gender", 
               category_orders={'education': education_order})
st.plotly_chart(fig10, use_container_width=True)

# End of the app

Overwriting Income_Analysis.py


In [395]:
! streamlit run Income_Analysis.py

^C


# Pipeline

In [472]:
df.to_csv('data.csv', index= False)

In [473]:
from sklearn.pipeline import Pipeline

In [474]:
#Splitting the data into input features & output

x = df.drop(['income', 'fnlwgt'], axis= 1)
df['income'] = df['income'].replace({'>50K': 1, '<=50K': 0})
y = df['income']
weights = df['fnlwgt']

C:\Users\Mariam Mahmoud\AppData\Local\Temp\ipykernel_14960\775587847.py:4: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



Numerical Pipeline

In [475]:
RC_cols = x.select_dtypes(exclude= 'object').columns.drop('has_capital_activity')
RC_cols

Index(['age', 'capital_gain', 'capital_loss', 'hours_per_week', 'net_capital'], dtype='object')

In [476]:
from sklearn.preprocessing import RobustScaler

RC_scaler = RobustScaler()

num_pipeline = Pipeline([('RobustScaler', RC_scaler)])
num_pipeline

Pipeline(steps=[('RobustScaler', RobustScaler())])

Categorical Pipeline

In [477]:
cat_cols = x.select_dtypes(include= 'object').columns
cat_cols

Index(['workclass', 'education', 'marital_status', 'occupation',
       'relationship', 'race', 'gender', 'region', 'employment_type'],
      dtype='object')

Cat_Pipeline1

In [478]:
x.education.unique()

array(['11th Grade', 'High School Graduate',
       "Associate's Degree (Academic)", 'College', '10th Grade',
       'Professional School', '7th to 8th Grade', "Bachelor's Degree",
       "Master's Degree", '5th to 6th Grade',
       "Associate's Degree (Vocational)", '9th Grade', 'Doctorate/PhD',
       '12th Grade', '1st to 4th Grade', 'Preschool'], dtype=object)

In [479]:
be_cols0 = df[['education']].columns
be_cols0

Index(['education'], dtype='object')

In [480]:
#Ordinal Pipeline
from category_encoders import BinaryEncoder

BE0  = BinaryEncoder()

BE_pipeline0 = Pipeline([('BE_Pipeline0', BE0)])
BE_pipeline0

Pipeline(steps=[('BE_Pipeline0', BinaryEncoder())])

Cat_Pipeline2 & Cat_Pipeline3

In [481]:
nominal_cols = x.select_dtypes(include = 'object').columns.drop('education')
nominal_cols

Index(['workclass', 'marital_status', 'occupation', 'relationship', 'race',
       'gender', 'region', 'employment_type'],
      dtype='object')

In [482]:
for col in nominal_cols:
  print(f'{col}: {x[col].nunique()}')
  print('-' * 20)

workclass: 8
--------------------
marital_status: 7
--------------------
occupation: 15
--------------------
relationship: 5
--------------------
race: 5
--------------------
gender: 2
--------------------
region: 6
--------------------
employment_type: 3
--------------------


In [483]:
x.workclass.value_counts(normalize = True) * 100

workclass
Private                             73.648019
Self-Employed (Not Incorporated)     8.355885
Local Government                     6.876359
State Government                     4.314360
Self-Employed (Incorporated)         3.622288
Federal Government                   3.114325
Without Pay                          0.046582
Never Worked                         0.022182
Name: proportion, dtype: float64

In [484]:
x.occupation.value_counts(normalize = True) * 100

occupation
Craft / Repair                  13.306863
Professional Specialty          13.280245
Executive / Managerial          13.220354
Administrative / Clerical       12.266537
Sales                           11.922719
Other Service                   10.625083
Machine Operator / Inspector     6.568032
Transport / Moving               5.135087
Handlers / Cleaners              4.529524
Farming / Fishing                3.271816
Technical Support                3.147598
Protective Service               2.162726
Private House Service            0.510181
Armed Forces                     0.031055
No Occupation                    0.022182
Name: proportion, dtype: float64

In [485]:
be_cols1 = df[['workclass']].columns
be_cols1

Index(['workclass'], dtype='object')

In [486]:
#Cat_Pipeline2

from sklearn.impute import SimpleImputer
from category_encoders import BinaryEncoder

imputer1 = SimpleImputer(strategy= 'most_frequent')
BE1 = BinaryEncoder()

BE_pipeline1 = Pipeline(steps= [ ('Constant Impute', imputer1),
                                        ('BinaryEncoder', BE1) ])
BE_pipeline1

Pipeline(steps=[('Constant Impute', SimpleImputer(strategy='most_frequent')),
                ('BinaryEncoder', BinaryEncoder())])

In [487]:
be_cols2 = df[['occupation']].columns
be_cols2

Index(['occupation'], dtype='object')

In [488]:
from category_encoders import BinaryEncoder

imputer2 = SimpleImputer(strategy= 'constant', fill_value= 'Other')
BE2 = BinaryEncoder()

BE_pipeline2 = Pipeline(steps= [ ('Constant Impute', imputer2),
                                        ('BinaryEncoder', BE2) ])
BE_pipeline2

Pipeline(steps=[('Constant Impute',
                 SimpleImputer(fill_value='Other', strategy='constant')),
                ('BinaryEncoder', BinaryEncoder())])

Cat_Pipeline4

In [489]:
ohe_cols = cat_cols.drop(['education', 'occupation', 'workclass'])
ohe_cols

Index(['marital_status', 'relationship', 'race', 'gender', 'region',
       'employment_type'],
      dtype='object')

In [490]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(drop= 'first', sparse_output= False)

ohe_pipeline = Pipeline(steps= [ ('OHE', ohe) ])
ohe_pipeline

Pipeline(steps=[('OHE', OneHotEncoder(drop='first', sparse_output=False))])

Assign each column to the corresponding pipeline


In [491]:
from sklearn.compose import ColumnTransformer

data_preprocessing = ColumnTransformer(transformers= [('Numerical Pipeline', num_pipeline, RC_cols),
                                                      ('BE_Pipeline0', BE_pipeline0, be_cols0),
                                                      ('BE_Pipeline1', BE_pipeline1, be_cols1),
                                                      ('BE_Pipeline2', BE_pipeline2, be_cols2),
                                                      ('OHE_Pipeline', ohe_pipeline, ohe_cols),
                                                     ],
                                  remainder= 'passthrough')
data_preprocessing

ColumnTransformer(remainder='passthrough',
                  transformers=[('Numerical Pipeline',
                                 Pipeline(steps=[('RobustScaler',
                                                  RobustScaler())]),
                                 Index(['age', 'capital_gain', 'capital_loss', 'hours_per_week', 'net_capital'], dtype='object')),
                                ('BE_Pipeline0',
                                 Pipeline(steps=[('BE_Pipeline0',
                                                  BinaryEncoder())]),
                                 Index(['education'], dtype='object')),
                                ('BE_Pipeline1',
                                 Pipeline(st...
                                 Pipeline(steps=[('Constant Impute',
                                                  SimpleImputer(fill_value='Other',
                                                                strategy='constant')),
                                                 ('BinaryEncoder',
                                                  BinaryEncoder())]),
                                 Index(['occupation'], dtype='object')),
                                ('OHE_Pipeline',
                                 Pipeline(steps=[('OHE',
                                                  OneHotEncoder(drop='first',
                                                                sparse_output=False))]),
                                 Index(['marital_status', 'relationship', 'race', 'gender', 'region',
       'employment_type'],
      dtype='object'))])

In [492]:
!pip install imbalanced-learn


# Modeling

In [ ]:
#Modelling
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier 

models = [
    ('Logistic Regression', LogisticRegression(random_state= 42, n_jobs= -1)),
    ('KNN', KNeighborsClassifier(n_jobs= -1)),
    ('Gaussian NB', GaussianNB()),
    ('Decision Tree', DecisionTreeClassifier(random_state= 42)),
    ('Random Forest', RandomForestClassifier(random_state= 42, n_jobs= -1)),
    ('SVC', SVC(random_state= 42)),
    ('XGBoost', XGBClassifier(random_state= 42)),
    ('LightGBM', LGBMClassifier(random_state= 42)),
]
smote = SMOTE()
for model in models:

    model_pipeline = Pipeline(steps= [ ('Preprocessing', data_preprocessing),
                                       ('SMOTE', smote),
                                       ('Model', model[1])])

    result = cross_validate(model_pipeline, x, y, cv= 5, scoring= 'f1', return_train_score= True, n_jobs= -1)

    print(model[0])
    print('Train F1 Score :', round(result['train_score'].mean() * 100, 2))
    print('Test F1 Score :', round(result['test_score'].mean() * 100, 2))
    print('-' * 50)

Logistic Regression
Train F1 Score : 66.59
Test F1 Score : 66.48
--------------------------------------------------
KNN
Train F1 Score : 76.37
Test F1 Score : 64.24
--------------------------------------------------
Gaussian NB
Train F1 Score : 53.89
Test F1 Score : 53.86
--------------------------------------------------
Decision Tree
Train F1 Score : 94.16
Test F1 Score : 61.77
--------------------------------------------------
Random Forest
Train F1 Score : 94.28
Test F1 Score : 67.78
--------------------------------------------------
SVC
Train F1 Score : 65.37
Test F1 Score : 65.2
--------------------------------------------------
XGBoost
Train F1 Score : 75.69
Test F1 Score : 72.3
--------------------------------------------------
LightGBM
Train F1 Score : 73.75
Test F1 Score : 72.28
--------------------------------------------------


In [493]:
#Hyperparameter Tuning for LightGBM (The top performing model) --> ROUND 1

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import RandomizedSearchCV
from lightgbm import LGBMClassifier


smote = SMOTE(random_state=42)

#Hyperparameter grids
grid = {
    'Model__learning_rate': [0.01, 0.05, 0.1],
    'Model__max_depth': [3, 5, 7],
    'Model__lambda_l2': [0, 1, 5],  
    'Model__lambda_l1': [0, 1, 5],
}

model_pipeline = Pipeline(steps=[('Preprocessing', data_preprocessing), ('SMOTE', smote), ('Model', LGBMClassifier(random_state=42))])

tuned_result = RandomizedSearchCV(model_pipeline, param_distributions=grid, scoring='f1', cv=5, n_jobs=-1, return_train_score=True, random_state=42)

tuned_result.fit(x, y)


    #Print best results
print(f"\n Best Parameters for the model:")
print(tuned_result.best_params_)
print()
print(f"Train F1 Score: {round(tuned_result.cv_results_['mean_train_score'][tuned_result.best_index_] * 100, 2)}")
print(f"Test F1 Score:  {round(tuned_result.best_score_ * 100, 2)}")

[LightGBM] [Warning] lambda_l1 is set=5, reg_alpha=0.0 will be ignored. Current value: lambda_l1=5
[LightGBM] [Warning] lambda_l2 is set=1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1
[LightGBM] [Warning] lambda_l1 is set=5, reg_alpha=0.0 will be ignored. Current value: lambda_l1=5
[LightGBM] [Warning] lambda_l2 is set=1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1
[LightGBM] [Info] Number of positive: 36377, number of negative: 36377
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.037974 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7625
[LightGBM] [Info] Number of data points in the train set: 72754, number of used features: 40
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No 

In [494]:
#Hyperparameter Tuning --> ROUND 2

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import RandomizedSearchCV
from lightgbm import LGBMClassifier

smote = SMOTE(random_state=42)

#Hyperparameter grids
grid = {
    'Model__learning_rate': [0.1, 0.15, 0.2],
    'Model__max_depth': [3, 5, 7],
    'Model__lambda_l2': [0, 1, 5],  
    'Model__lambda_l1': [5, 6, 7],
    }


model_pipeline = Pipeline(steps=[('Preprocessing', data_preprocessing), ('SMOTE', smote), ('Model', LGBMClassifier(random_state=42))])

tuned_result = RandomizedSearchCV(model_pipeline, param_distributions=grid, scoring='f1', cv=5, n_jobs=-1, return_train_score=True, random_state=42)

tuned_result.fit(x, y)


    #Print best results
print(f"\n Best Parameters for the model:")
print(tuned_result.best_params_)
print()
print(f"Train F1 Score: {round(tuned_result.cv_results_['mean_train_score'][tuned_result.best_index_] * 100, 2)}")
print(f"Test F1 Score:  {round(tuned_result.best_score_ * 100, 2)}")


[LightGBM] [Warning] lambda_l1 is set=7, reg_alpha=0.0 will be ignored. Current value: lambda_l1=7
[LightGBM] [Warning] lambda_l2 is set=1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1
[LightGBM] [Warning] lambda_l1 is set=7, reg_alpha=0.0 will be ignored. Current value: lambda_l1=7
[LightGBM] [Warning] lambda_l2 is set=1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1
[LightGBM] [Info] Number of positive: 36377, number of negative: 36377
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.066632 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7625
[LightGBM] [Info] Number of data points in the train set: 72754, number of used features: 40
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No 

In [495]:
#Hyperparameter Tuning --> ROUND 3

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import RandomizedSearchCV
from lightgbm import LGBMClassifier

smote = SMOTE(random_state=42)

#Hyperparameter grids
grid = {
    'Model__learning_rate': [0.2, 0.25, 0.3],
    'Model__max_depth': [3, 5, 7],
    'Model__lambda_l2': [0, 1, 5],  
    'Model__lambda_l1': [7, 8, 9],
    }


model_pipeline = Pipeline(steps=[('Preprocessing', data_preprocessing), ('SMOTE', smote), ('Model', LGBMClassifier(random_state=42))])

tuned_result = RandomizedSearchCV(model_pipeline, param_distributions=grid, scoring='f1', cv=5, n_jobs=-1, return_train_score=True, random_state=42)

tuned_result.fit(x, y)


    #Print best results
print(f"\n Best Parameters for the model:")
print(tuned_result.best_params_)
print()
print(f"Train F1 Score: {round(tuned_result.cv_results_['mean_train_score'][tuned_result.best_index_] * 100, 2)}")
print(f"Test F1 Score:  {round(tuned_result.best_score_ * 100, 2)}")


[LightGBM] [Warning] lambda_l1 is set=7, reg_alpha=0.0 will be ignored. Current value: lambda_l1=7
[LightGBM] [Warning] lambda_l2 is set=0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0
[LightGBM] [Warning] lambda_l1 is set=7, reg_alpha=0.0 will be ignored. Current value: lambda_l1=7
[LightGBM] [Warning] lambda_l2 is set=0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0
[LightGBM] [Info] Number of positive: 36377, number of negative: 36377
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009759 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7625
[LightGBM] [Info] Number of data points in the train set: 72754, number of used features: 40
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No 

In [496]:
#Hyperparameter Tuning --> ROUND 4

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import RandomizedSearchCV
from lightgbm import LGBMClassifier

smote = SMOTE(random_state=42)

#Hyperparameter grids
grid = {
    'Model__learning_rate': [0.2, 0.25, 0.3],
    'Model__max_depth': [3, 5, 7],
    'Model__lambda_l2': [-1, 0, 1],  
    'Model__lambda_l1': [-1, 7, 8],
    }


model_pipeline = Pipeline(steps=[('Preprocessing', data_preprocessing), ('SMOTE', smote), ('Model', LGBMClassifier(random_state=42))])

tuned_result = RandomizedSearchCV(model_pipeline, param_distributions=grid, scoring='f1', cv=5, n_jobs=-1, return_train_score=True, random_state=42)

tuned_result.fit(x, y)


    #Print best results
print(f"\n Best Parameters for the model:")
print(tuned_result.best_params_)
print()
print(f"Train F1 Score: {round(tuned_result.cv_results_['mean_train_score'][tuned_result.best_index_] * 100, 2)}")
print(f"Test F1 Score:  {round(tuned_result.best_score_ * 100, 2)}")


c:\Users\Mariam Mahmoud\anaconda3\envs\ml\Lib\site-packages\sklearn\model_selection\_validation.py:528: FitFailedWarning:


45 fits failed out of a total of 50.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
15 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\Mariam Mahmoud\anaconda3\envs\ml\Lib\site-packages\sklearn\model_selection\_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Mariam Mahmoud\anaconda3\envs\ml\Lib\site-packages\sklearn\base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Users\Mariam Mahmoud\anaconda3\envs\ml\Lib\site-packages

[LightGBM] [Warning] lambda_l1 is set=8, reg_alpha=0.0 will be ignored. Current value: lambda_l1=8
[LightGBM] [Warning] lambda_l2 is set=0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0
[LightGBM] [Warning] lambda_l1 is set=8, reg_alpha=0.0 will be ignored. Current value: lambda_l1=8
[LightGBM] [Warning] lambda_l2 is set=0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0
[LightGBM] [Info] Number of positive: 36377, number of negative: 36377
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.024086 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7625
[LightGBM] [Info] Number of data points in the train set: 72754, number of used features: 40
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No 

### 🔍 Hyperparameter Tuning Results

There was **no further improvement** achieved through hyperparameter tuning, as the **default model already yielded the highest performance**.

➡️ Next step: **Apply feature selection** to explore potential enhancements in model performance.


In [508]:
#Feature Selection: Wrapper Method: KBest
from sklearn.feature_selection import SelectKBest

KBest = SelectKBest()

KBest_pipeline = Pipeline(steps= [ ('Preprocessing', data_preprocessing),
                                    ('KBest', KBest) ])

KBest_pipeline.fit(x, y)

c:\Users\Mariam Mahmoud\anaconda3\envs\ml\Lib\site-packages\sklearn\compose\_column_transformer.py:1667: FutureWarning:


The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).




Pipeline(steps=[('Preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('Numerical Pipeline',
                                                  Pipeline(steps=[('RobustScaler',
                                                                   RobustScaler())]),
                                                  Index(['age', 'capital_gain', 'capital_loss', 'hours_per_week', 'net_capital'], dtype='object')),
                                                 ('BE_Pipeline0',
                                                  Pipeline(steps=[('BE_Pipeline0',
                                                                   BinaryEncoder())]),
                                                  Index(['education'], dtype='obje...
                                                                   'Impute',
                                                                   SimpleImputer(fill_value='Other',
                                                                                 strategy='constant')),
                                                                  ('BinaryEncoder',
                                                                   BinaryEncoder())]),
                                                  Index(['occupation'], dtype='object')),
                                                 ('OHE_Pipeline',
                                                  Pipeline(steps=[('OHE',
                                                                   OneHotEncoder(drop='first',
                                                                                 sparse_output=False))]),
                                                  Index(['marital_status', 'relationship', 'race', 'gender', 'region',
       'employment_type'],
      dtype='object'))])),
                ('KBest', SelectKBest())])

In [514]:
#Get the feature names after preprocessing
feature_names = KBest_pipeline.named_steps['Preprocessing'].get_feature_names_out()
feature_names

array(['Numerical Pipeline__age', 'Numerical Pipeline__capital_gain',
       'Numerical Pipeline__capital_loss',
       'Numerical Pipeline__hours_per_week',
       'Numerical Pipeline__net_capital', 'BE_Pipeline0__education_0',
       'BE_Pipeline0__education_1', 'BE_Pipeline0__education_2',
       'BE_Pipeline0__education_3', 'BE_Pipeline0__education_4',
       'BE_Pipeline1__0_0', 'BE_Pipeline1__0_1', 'BE_Pipeline1__0_2',
       'BE_Pipeline1__0_3', 'BE_Pipeline2__0_0', 'BE_Pipeline2__0_1',
       'BE_Pipeline2__0_2', 'BE_Pipeline2__0_3', 'BE_Pipeline2__0_4',
       'OHE_Pipeline__marital_status_Married (Armed Forces Spouse)',
       'OHE_Pipeline__marital_status_Married (Civilian Spouse)',
       'OHE_Pipeline__marital_status_Married, Spouse Absent',
       'OHE_Pipeline__marital_status_Never Married',
       'OHE_Pipeline__marital_status_Separated',
       'OHE_Pipeline__marital_status_Widowed',
       'OHE_Pipeline__relationship_Not-in-family',
       'OHE_Pipeline__relationship_

In [516]:
#Get the scores calculated by SelectKBest
scores = KBest_pipeline.named_steps['KBest'].scores_
scores

array([2.67066201e+03, 2.49114181e+03, 1.05330246e+03, 2.62761350e+03,
       2.29085539e+03, 2.23312183e+01, 2.44838420e+03, 1.50311232e+02,
       6.56975551e+02, 7.82490114e+01, 3.14557745e+00, 7.60102883e+02,
       5.60609540e+02, 8.46131500e+02, 3.14557745e+00, 1.77572533e+02,
       1.92992138e+02, 4.25778499e+02, 2.24524700e+01, 3.93846569e+00,
       1.18789796e+04, 7.00856002e+01, 5.41424169e+03, 2.58274041e+02,
       2.02128736e+02, 1.79613796e+03, 3.41315404e+02, 2.58836910e+03,
       1.00678337e+03, 1.10389006e+01, 3.94895415e+02, 2.83538017e+01,
       3.39665937e+02, 2.30843084e+03, 1.34838618e+01, 3.02057200e+02,
       5.79669854e+00, 2.44476572e+01, 7.95053519e+01, 2.42692301e+03,
       1.63810808e+03, 4.81810021e+03])

In [518]:
#Build the dataframe
importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': scores})

importance_df = importance_df.sort_values(by='Importance', ascending=False)
importance_df


,Feature,Importance
20,OHE_Pipeline__marital_status_Married (Civilian...,11878.979608
22,OHE_Pipeline__marital_status_Never Married,5414.241694
41,remainder__has_capital_activity,4818.100207
0,Numerical Pipeline__age,2670.662013
3,Numerical Pipeline__hours_per_week,2627.613500
27,OHE_Pipeline__relationship_Own-child,2588.369103
1,Numerical Pipeline__capital_gain,2491.141813
6,BE_Pipeline0__education_1,2448.384202
39,OHE_Pipeline__employment_type_Overtime,2426.923014
33,OHE_Pipeline__gender_Male,2308.430841


In [520]:
#Normalized dataframe
importance_df['Normalized_Importance'] = (importance_df['Importance'] / importance_df['Importance'].sum()) * 100

importance_df = importance_df.sort_values(by='Normalized_Importance', ascending=False)
importance_df

,Feature,Importance,Normalized_Importance
20,OHE_Pipeline__marital_status_Married (Civilian...,11878.979608,22.231617
22,OHE_Pipeline__marital_status_Never Married,5414.241694,10.132802
41,remainder__has_capital_activity,4818.100207,9.017118
0,Numerical Pipeline__age,2670.662013,4.998168
3,Numerical Pipeline__hours_per_week,2627.613500,4.917602
27,OHE_Pipeline__relationship_Own-child,2588.369103,4.844156
1,Numerical Pipeline__capital_gain,2491.141813,4.662194
6,BE_Pipeline0__education_1,2448.384202,4.582173
39,OHE_Pipeline__employment_type_Overtime,2426.923014,4.542008
33,OHE_Pipeline__gender_Male,2308.430841,4.320249


**After identifying the most influential features, we will pass them to the modeling pipeline and select the top 25 features for training.**

In [549]:
#Modelling
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.feature_selection import SelectKBest,  f_classif
from sklearn.model_selection import cross_validate
from lightgbm import LGBMClassifier 


smote = SMOTE()

model_pipeline = Pipeline(steps= [ ('Preprocessing', data_preprocessing),
                                       ('SMOTE', smote),
                                        ('KBest', SelectKBest(score_func=f_classif, k=25)),
                                       ('Model', LGBMClassifier(random_state= 42))])

result = cross_validate(model_pipeline, x, y, cv= 5, scoring= 'f1', return_train_score= True, n_jobs= -1)

print(model[0])
print('Train F1 Score :', round(result['train_score'].mean() * 100, 2))
print('Test F1 Score :', round(result['test_score'].mean() * 100, 2))


ColumnTransformer(remainder='passthrough',
                  transformers=[('Numerical Pipeline',
                                 Pipeline(steps=[('RobustScaler',
                                                  RobustScaler())]),
                                 Index(['age', 'capital_gain', 'capital_loss', 'hours_per_week', 'net_capital'], dtype='object')),
                                ('Ordinal_Pipeline',
                                 Pipeline(steps=[('OrdinalEncoder',
                                                  OrdinalEncoder(categories=[['Preschool',
                                                                              '1st '
                                                                              'to '
                                                                              '4th '
                                                                              'Grade',
                                                                              '5th 

**After passing the most influential features to the model only, the model performance also worsened, so will proceed with the defaullt parameters.**

In [551]:
#fitting the model with default paarmeters
 
from imblearn.pipeline import Pipeline
from lightgbm import LGBMClassifier

LGBM_pipeline = Pipeline(steps= [  ('Preprocessing', data_preprocessing),
                                    ('SMOTE', smote),
                                    ('Model', LGBMClassifier(random_state= 42))])

LGBM_pipeline.fit(x, y)

[LightGBM] [Info] Number of positive: 36377, number of negative: 36377
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006703 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7634
[LightGBM] [Info] Number of data points in the train set: 72754, number of used features: 40
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


c:\Users\Mariam Mahmoud\anaconda3\envs\ml\Lib\site-packages\sklearn\compose\_column_transformer.py:1667: FutureWarning:


The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).




Pipeline(steps=[('Preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('Numerical Pipeline',
                                                  Pipeline(steps=[('RobustScaler',
                                                                   RobustScaler())]),
                                                  Index(['age', 'capital_gain', 'capital_loss', 'hours_per_week', 'net_capital'], dtype='object')),
                                                 ('BE_Pipeline0',
                                                  Pipeline(steps=[('BE_Pipeline0',
                                                                   BinaryEncoder())]),
                                                  Index(['education'], dtype='obje...
                                                                                 strategy='constant')),
                                                                  ('BinaryEncoder',
                                                                   BinaryEncoder())]),
                                                  Index(['occupation'], dtype='object')),
                                                 ('OHE_Pipeline',
                                                  Pipeline(steps=[('OHE',
                                                                   OneHotEncoder(drop='first',
                                                                                 sparse_output=False))]),
                                                  Index(['marital_status', 'relationship', 'race', 'gender', 'region',
       'employment_type'],
      dtype='object'))])),
                ('SMOTE', SMOTE()),
                ('Model', LGBMClassifier(random_state=42))])

In [552]:
LGBM_pipeline.predict(x.head(1))[0]

c:\Users\Mariam Mahmoud\anaconda3\envs\ml\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning:

X does not have valid feature names, but LGBMClassifier was fitted with feature names



np.int64(0)

## Deployment

In [246]:
! pip install joblib

In [532]:
import joblib

joblib.dump(LGBM_pipeline, 'LGBM.pkl')

['LGBM.pkl']

In [533]:
l1 = df[['capital_gain', 'capital_loss', 'net_capital']]

for col_name in l1:
    print(col_name)
    print(df[col_name].max())
    print(df[col_name].min())

capital_gain
99999
0
capital_loss
4356
0
net_capital
99999
-4356


In [536]:
%%writefile Income_Prediction.py
import streamlit as st
import pandas as pd
import joblib
from pathlib import Path

st.set_page_config(layout='wide', page_title='Income Classification App', page_icon='💰')

st.markdown("""
    <h1 style='text-align:center; color:#2C3E50;'>💰 Income Classification App</h1>
    <p style='text-align:center; color:gray;'>Predict whether a person earns above or below a certain income threshold</p>
""", unsafe_allow_html=True)

# --- Load Data ---
data_path = Path("data.csv")
model_path = Path("LGBM.pkl")

if not data_path.exists():
    st.error("❌ 'data.csv' not found. Please make sure it’s in the same folder as this script.")
    st.stop()

if not model_path.exists():
    st.error("❌ 'LGBM.pkl' model file not found. Please make sure it’s in the same folder as this script.")
    st.stop()

df = pd.read_csv(data_path)

with st.expander("📊 Preview Dataset"):
    st.dataframe(df.head())

#Initialize session state
for key in ['net_manual', 'emp_manual']:
    if key not in st.session_state:
        st.session_state[key] = None

#Sidebar Inputs
st.sidebar.header("⚙️ Numeric Inputs")

age = st.sidebar.slider("Age", 17, 90, 38)
capital_gain = st.sidebar.number_input("Capital Gain in USD", min_value=0.0, max_value=99999.0, value=0.0, step=100.0)
capital_loss = st.sidebar.number_input("Capital Loss in USD", min_value=0.0, max_value=4356.0, value=0.0, step=100.0)
hours_per_week = st.sidebar.slider("Hours per Week", 1, 99, 40)

#Net Capital
if capital_gain == 0 and capital_loss == 0:
    net_capital = st.sidebar.number_input("Net Capital in USD", min_value=-4356.0, max_value=99999.0, value=0.0, step = 100.0)
else:
    net_capital = capital_gain - capital_loss

st.session_state['net_manual'] = net_capital

st.info(f"💡 Net Capital in USD= {net_capital} (Calculated from gain and loss)")

#Has Capital Activity
if capital_gain != 0 or capital_loss != 0 or net_capital != 0:
    has_capital_activity = "Yes"
else:
    has_capital_activity = "No"
st.caption(f"Capital Activity: **{has_capital_activity}**")

#Employment Type
if hours_per_week < 35:
    default_emp = "Part_time"
elif hours_per_week <= 45:
    default_emp = "Full_time"
else:
    default_emp = "Overtime"

st.session_state['emp_manual'] = default_emp if st.session_state['emp_manual'] is None else st.session_state['emp_manual']

employment_type = st.sidebar.selectbox(
    "Employment Type",
    ['Part_time', 'Full_time', 'Overtime'],
    index=['Part_time', 'Full_time', 'Overtime'].index(st.session_state['emp_manual']),
    key='emp_manual'
)

#Main Page Layout
st.markdown("### 🧠 Categorical Inputs")
col1, col2, col3 = st.columns(3)

with col1:
    workclass = st.selectbox("Workclass", df['workclass'].dropna().unique())
    education = st.selectbox("Education", df['education'].dropna().unique())
    education_order = {'Preschool':1,
                       '1st to 4th Grade' :2,
                       '5th to 6th Grade': 3,
                       '7th to 8th Grade':4, 
                       '9th Grade': 5,
                       '10th Grade': 6,
                       '11th Grade': 7, '12th Grade': 8,
                       'High School Graduate': 9, 
                       'College':10, 
                       "Associate's Degree (Academic)":11,
                       "Associate's Degree (Vocational)":12,
                       "Bachelor's Degree":13,  
                       "Master's Degree":14,
                       'Professional School':15, 'Doctorate/PhD':16}

    # Convert education string → ordinal numeric value
    education_encoded = education_order.get(education, None)

    if education_encoded is None:
        st.error("Unknown education category selected — please verify the mapping.")
        st.stop()

    marital_status = st.selectbox("Marital Status", df['marital_status'].dropna().unique())
   

with col2:
    occupation = st.selectbox("Occupation", df['occupation'].dropna().unique())
    relationship = st.selectbox("Relationship", df['relationship'].dropna().unique())
    race = st.selectbox("Race", df['race'].dropna().unique())

with col3:
    gender = st.selectbox("Gender", df['gender'].dropna().unique())
    region = st.selectbox("Region", df['region'].dropna().unique())

#Load Model
model = joblib.load('LGBM.pkl')


input_cols = df.drop(['income', 'fnlwgt'], axis=1).columns

new_data = pd.DataFrame(columns=input_cols, data =[[age, workclass, education, marital_status,
                                                    occupation, relationship, race, gender, capital_gain,
                                                    capital_loss, hours_per_week, region, net_capital,
                                                    1 if has_capital_activity == 'Yes' else 0,                                                  
                                                    employment_type]])

#Prediction
st.markdown("## 🔍 Predict Income Level")

if st.button('Predict Income'):
    try:
        result = model.predict(new_data)[0]
        if result == 0:
            st.success("💼 Prediction: **Low Income (≤ $50k)**")
        else:
            st.warning("💰 Prediction: **High Income (> $50k)**")
    except Exception as e:
        st.error(f"Prediction failed: {e}")

Overwriting Income_Prediction.py


In [535]:
!streamlit run Income_Prediction.py

^C


In [554]:
!pip install pipreqs

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13775 sha256=3cc61e1e8657be6a4fb1e941dff037c5c8de5aecc3af31bad21cf07a721b598e
  Stored in directory: c:\users\mariam mahmoud\appdata\local\pip\cache\wheels\0b\1d\03\175286677fb5a1341cc3e4755bf8ec0ed08f3329afd67446b0
Successfully built docopt

   -------------------------- ------------- 2/3 [pipreqs]
   ---------------------------------------- 3/3 [pipreqs]



  DEPRECATION: Building 'docopt' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'docopt'. Discussion can be found at https://github.com/pypa/pip/issues/6334


In [556]:
import pipreqs

! pipreqs .

Please, verify manually the final list of requirements.txt to avoid possible dependency confusions.
INFO: Successfully saved requirements file in .\requirements.txt
